# Social Media Content Engagement Analysis
**Project ID:** DA-020 | **Course:** Digital Marketing

**Objective:** Analyze post-level performance, compare post formats and publishing periods, rank posts, visualize results, and propose content experiments.

> The included CSV is synthetic demonstration data. Do not treat it as real brand or platform performance.

## 1. Import libraries and load data
Run this notebook from the repository folder. If using Colab, upload `social_media_posts.csv` to the session.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_FILE = "social_media_posts.csv"
df = pd.read_csv(DATA_FILE)

print("Dataset shape:", df.shape)
display(df.head())

## 2. Understand the dataset
Check data types, missing values, duplicate IDs, and basic numeric summaries before calculating metrics.

In [ ]:
print(df.info())
print("\\nMissing values by column:")
display(df.isna().sum().to_frame("missing_count"))
print("Duplicate post IDs:", df["post_id"].duplicated().sum())
display(df.describe(include="all").T)

## 3. Clean and prepare data
Dates and metric fields are converted explicitly. Missing likes/comments/shares are treated as zero for additive engagement counts. Missing reach or impressions remain missing because they are denominators and cannot safely be invented.

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

metric_cols = ["impressions", "reach", "likes", "comments", "shares"]
for col in metric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Engagement components: missing component values are treated as zero.
engagement_cols = ["likes", "comments", "shares"]
df[engagement_cols] = df[engagement_cols].fillna(0)

# Remove duplicate post identifiers; keep the first record.
df = df.drop_duplicates(subset="post_id", keep="first").copy()

# Avoid negative counts if malformed records are introduced.
for col in metric_cols:
    df.loc[df[col] < 0, col] = np.nan

print("Rows after cleaning:", len(df))
display(df.isna().sum().to_frame("missing_after_cleaning"))

## 4. Calculate engagement metrics
**Total engagements** = likes + comments + shares. **ERR** = total engagements / reach × 100. **ERI** = total engagements / impressions × 100. **Reach rate** = reach / impressions × 100. Rates are set to missing where the denominator is missing or zero.

In [ ]:
df["total_engagements"] = df[engagement_cols].sum(axis=1)

df["engagement_rate_reach_pct"] = np.where(
    df["reach"].notna() & (df["reach"] > 0),
    df["total_engagements"] / df["reach"] * 100,
    np.nan
)

df["engagement_rate_impressions_pct"] = np.where(
    df["impressions"].notna() & (df["impressions"] > 0),
    df["total_engagements"] / df["impressions"] * 100,
    np.nan
)

df["reach_rate_pct"] = np.where(
    df["impressions"].notna() & (df["impressions"] > 0) & df["reach"].notna(),
    df["reach"] / df["impressions"] * 100,
    np.nan
)

display(df[["post_id", "total_engagements", "engagement_rate_reach_pct",
            "engagement_rate_impressions_pct", "reach_rate_pct"]].head(10))

## 5. Compare post formats and publishing periods
The tables use the mean and median ERR, number of posts, and average reach. Always read the denominator definition alongside the percentage.

In [ ]:
by_type = (df.groupby("post_type")
           .agg(posts=("post_id", "count"),
                avg_ERR_pct=("engagement_rate_reach_pct", "mean"),
                median_ERR_pct=("engagement_rate_reach_pct", "median"),
                avg_reach=("reach", "mean"))
           .sort_values("avg_ERR_pct", ascending=False)
           .round(2))
display(by_type)

by_period = (df.groupby("posting_period")
             .agg(posts=("post_id", "count"),
                  avg_ERR_pct=("engagement_rate_reach_pct", "mean"),
                  median_ERR_pct=("engagement_rate_reach_pct", "median"),
                  avg_reach=("reach", "mean"))
             .sort_values("avg_ERR_pct", ascending=False)
             .round(2))
display(by_period)

by_platform = (df.groupby("platform")
               .agg(posts=("post_id", "count"),
                    avg_ERR_pct=("engagement_rate_reach_pct", "mean"),
                    avg_reach=("reach", "mean"))
               .sort_values("avg_ERR_pct", ascending=False)
               .round(2))
display(by_platform)

## 6. Rank high-performing posts
Posts with a valid ERR are ranked from highest to lowest. Consider checking reach and total engagements too: a high rate on a small audience may not create the largest overall impact.

In [ ]:
ranking = (df.dropna(subset=["engagement_rate_reach_pct"])
           .sort_values("engagement_rate_reach_pct", ascending=False)
           [["post_id", "date", "platform", "post_type", "posting_period",
             "reach", "total_engagements", "engagement_rate_reach_pct"]]
           .head(15)
           .round(2))
display(ranking)

## 7. Visualizations
Four charts are required for this project. They are also saved to `outputs/` for use in a report.

In [ ]:
os.makedirs("outputs", exist_ok=True)

# Chart 1: average ERR by post type
plt.figure(figsize=(9, 5))
order = by_type.index
sns.barplot(data=df, x="post_type", y="engagement_rate_reach_pct",
            order=order, errorbar=None)
plt.title("Average Engagement Rate by Reach — Post Type")
plt.xlabel("Post type")
plt.ylabel("Average ERR (%)")
plt.tight_layout()
plt.savefig("outputs/01_err_by_post_type.png", dpi=150)
plt.show()

In [ ]:
# Chart 2: average ERR by publishing period
period_order = ["Morning", "Afternoon", "Evening", "Night"]
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="posting_period", y="engagement_rate_reach_pct",
            order=period_order, errorbar=None)
plt.title("Average Engagement Rate by Reach — Posting Period")
plt.xlabel("Posting period")
plt.ylabel("Average ERR (%)")
plt.tight_layout()
plt.savefig("outputs/02_err_by_posting_period.png", dpi=150)
plt.show()

In [ ]:
# Chart 3: daily average ERR trend
daily = (df.dropna(subset=["engagement_rate_reach_pct"])
         .groupby("date", as_index=False)["engagement_rate_reach_pct"].mean())
plt.figure(figsize=(11, 5))
sns.lineplot(data=daily, x="date", y="engagement_rate_reach_pct", marker="o")
plt.title("Daily Average Engagement Rate by Reach")
plt.xlabel("Date")
plt.ylabel("Average ERR (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("outputs/03_daily_err_trend.png", dpi=150)
plt.show()

In [ ]:
# Chart 4: average ERR by post type and platform
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x="post_type", y="engagement_rate_reach_pct",
            hue="platform", errorbar=None)
plt.title("Average Engagement Rate by Post Type and Platform")
plt.xlabel("Post type")
plt.ylabel("Average ERR (%)")
plt.legend(title="Platform")
plt.tight_layout()
plt.savefig("outputs/04_err_type_platform.png", dpi=150)
plt.show()

## 8. Findings generated from this dataset
The statements below are computed from the data loaded in this notebook. Since the CSV is synthetic, use them only to demonstrate how findings should be reported.

In [ ]:
best_type = by_type["avg_ERR_pct"].idxmax()
best_period = by_period["avg_ERR_pct"].idxmax()
best_platform = by_platform["avg_ERR_pct"].idxmax()
top_post = ranking.iloc[0] if len(ranking) else None

print(f"Highest average ERR by post type: {best_type} "
      f"({by_type.loc[best_type, 'avg_ERR_pct']:.2f}%).")
print(f"Highest average ERR by posting period: {best_period} "
      f"({by_period.loc[best_period, 'avg_ERR_pct']:.2f}%).")
print(f"Highest average ERR by platform: {best_platform} "
      f"({by_platform.loc[best_platform, 'avg_ERR_pct']:.2f}%).")

if top_post is not None:
    print(f"Top post: {top_post['post_id']} | {top_post['post_type']} | "
          f"{top_post['posting_period']} | ERR "
          f"{top_post['engagement_rate_reach_pct']:.2f}%")

## 9. Content experiments to consider

1. **Format experiment:** Compare carousel and reel posts using similar topics and creative effort. Primary KPI: average ERR.
2. **Timing experiment:** Rotate comparable posts between morning and evening. Primary KPI: average ERR; secondary KPI: reach.
3. **Call-to-action experiment:** Compare a question-based caption with a neutral caption. KPI: comments per 1,000 reached.
4. **Platform experiment:** Adapt the same content idea for two platforms. Compare ERR separately by platform.

Run tests over multiple weeks, use comparable topics, and avoid changing many variables at once. Group averages are descriptive and do not prove causation.

## 10. Export the cleaned analysis

The following line saves calculated metrics for later reporting. Do not upload private account exports to a public repository.

In [ ]:
df.to_csv("outputs/cleaned_posts_with_metrics.csv", index=False)
print("Saved cleaned data and four charts in the outputs/ folder.")

## 11. Conclusion

This notebook demonstrates data import, inspection, cleaning, calculated columns, percentage metrics, grouped comparisons, ranking, visualization, and experiment design. The engagement-rate denominator is explicitly documented to make reporting transparent.